In [7]:
import os
import cv2
import pickle

IMAGE_DIR="dataset/images/train"
LABEL_DIR="dataset/labels/train"
OUTPUT_FILE="ground_truth.pkl"

dataset={}

image_files=sorted([
f for f in os.listdir(IMAGE_DIR)
if f.endswith(".jpg")
])

class_names={
0:"text",
1:"title",
2:"list",
3:"table",
4:"figure"
}

for image_name in image_files:

    image_path=os.path.join(
        IMAGE_DIR,
        image_name
    )

    image=cv2.imread(image_path)

    if image is None:
        continue

    h,w=image.shape[:2]

    label_path=os.path.join(
        LABEL_DIR,
        image_name.replace(".jpg",".txt")
    )

    objects=[]

    if os.path.exists(label_path):

        with open(label_path,"r") as f:

            for line in f:

                cls,xc,yc,bw,bh=map(
                    float,
                    line.strip().split()
                )

                x1=(xc-bw/2)*w
                y1=(yc-bh/2)*h

                x2=(xc+bw/2)*w
                y2=(yc+bh/2)*h

                objects.append({

                    "class_id":int(cls),

                    "class_name":class_names[int(cls)],

                    "bbox":[
                        int(x1),
                        int(y1),
                        int(x2),
                        int(y2)
                    ]

                })

    dataset[image_name]={

        "image_path":image_path,

        "width":w,

        "height":h,

        "objects":objects

    }

    print(
        image_name,
        len(objects)
    )

with open(
    OUTPUT_FILE,
    "wb"
) as f:

    pickle.dump(
        dataset,
        f
    )

print()

print("Saved",OUTPUT_FILE)
print("Images",len(dataset))

0.jpg 14
10.jpg 6
100.jpg 16
1001.jpg 8
1004.jpg 8
1005.jpg 13
1006.jpg 18
1007.jpg 9
1008.jpg 10
1010.jpg 8
1012.jpg 8
1013.jpg 8
1015.jpg 9
1016.jpg 3
1017.jpg 14
1019.jpg 7
102.jpg 10
1020.jpg 8
1021.jpg 9
1022.jpg 8
1023.jpg 13
1025.jpg 11
1026.jpg 11
1027.jpg 9
1028.jpg 12
1029.jpg 9
103.jpg 9
1030.jpg 11
1031.jpg 2
1033.jpg 3
1034.jpg 6
1035.jpg 9
1037.jpg 19
1038.jpg 4
1039.jpg 11
104.jpg 12
1040.jpg 10
1041.jpg 12
1042.jpg 2
1044.jpg 6
1045.jpg 9
1047.jpg 19
1048.jpg 2
1049.jpg 5
105.jpg 11
1050.jpg 4
1051.jpg 7
1053.jpg 15
1054.jpg 9
1057.jpg 13
1058.jpg 11
1059.jpg 12
106.jpg 8
1060.jpg 12
1061.jpg 12
1062.jpg 9
1063.jpg 6
1064.jpg 19
1065.jpg 2
1066.jpg 12
1067.jpg 10
1068.jpg 14
1069.jpg 15
107.jpg 9
1071.jpg 9
1072.jpg 7
1073.jpg 12
1074.jpg 4
1075.jpg 10
1076.jpg 8
1078.jpg 8
108.jpg 10
1080.jpg 5
1082.jpg 9
1083.jpg 2
1085.jpg 11
1086.jpg 22
1087.jpg 11
1088.jpg 10
1089.jpg 7
1092.jpg 2
1093.jpg 2
1094.jpg 12
1095.jpg 11
1096.jpg 8
1098.jpg 8
1099.jpg 14
11.jpg 14
1100.j

In [11]:
import cv2
import pickle

GROUND_TRUTH_FILE="ground_truth.pkl"
OUTPUT_FILE="region_proposals.pkl"

with open(GROUND_TRUTH_FILE,"rb") as f:
    dataset=pickle.load(f)

ss=cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()

proposal_dataset={}

for image_name in list(dataset.keys())[:10]:
    image_path=dataset[image_name]["image_path"]

    image=cv2.imread(image_path)

    if image is None:
        continue

    ss.setBaseImage(image)

    ss.switchToSelectiveSearchFast()

    rects=ss.process()

    proposals=[]

    for (x,y,w,h) in rects:

        proposals.append({

            "bbox":[
                int(x),
                int(y),
                int(x+w),
                int(y+h)
            ]

        })

    proposal_dataset[image_name]={

        "image_path":image_path,

        "proposals":proposals

    }

    print(
        image_name,
        len(proposals)
    )

with open(
    OUTPUT_FILE,
    "wb"
) as f:

    pickle.dump(
        proposal_dataset,
        f
    )

print()

print("Saved",OUTPUT_FILE)
print("Images",len(proposal_dataset))

first_image=list(proposal_dataset.keys())[0]

image=cv2.imread(
    proposal_dataset[first_image]["image_path"]
)

output=image.copy()

for proposal in proposal_dataset[first_image]["proposals"][:100]:

    x1,y1,x2,y2=proposal["bbox"]

    cv2.rectangle(
        output,
        (x1,y1),
        (x2,y2),
        (0,255,0),
        1
    )

cv2.imshow(
    "Selective Search",
    output
)

cv2.waitKey(0)

cv2.destroyAllWindows()

0.jpg 2626
10.jpg 905
100.jpg 2743
1001.jpg 2642
1004.jpg 2908
1005.jpg 3213
1006.jpg 2757
1007.jpg 3130
1008.jpg 2786
1010.jpg 2947

Saved region_proposals.pkl
Images 10


In [12]:
import pickle

GROUND_TRUTH_FILE="ground_truth.pkl"
PROPOSAL_FILE="region_proposals.pkl"
OUTPUT_FILE="training_samples.pkl"

with open(GROUND_TRUTH_FILE,"rb") as f:
    ground_truth=pickle.load(f)

with open(PROPOSAL_FILE,"rb") as f:
    proposals=pickle.load(f)

def compute_iou(boxA,boxB):

    xA=max(boxA[0],boxB[0])
    yA=max(boxA[1],boxB[1])

    xB=min(boxA[2],boxB[2])
    yB=min(boxA[3],boxB[3])

    inter=max(0,xB-xA)*max(0,yB-yA)

    areaA=(boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    areaB=(boxB[2]-boxB[0])*(boxB[3]-boxB[1])

    union=areaA+areaB-inter

    if union==0:
        return 0

    return inter/union

training_samples={}

for image_name in proposals.keys():

    gt_objects=ground_truth[image_name]["objects"]

    proposal_list=proposals[image_name]["proposals"]

    positives=[]
    negatives=[]

    for proposal in proposal_list:

        proposal_box=proposal["bbox"]

        best_iou=0
        best_gt=None

        for gt in gt_objects:

            gt_box=gt["bbox"]

            iou=compute_iou(
                proposal_box,
                gt_box
            )

            if iou>best_iou:

                best_iou=iou
                best_gt=gt

        if best_iou>=0.5:

            positives.append({

                "proposal_box":proposal_box,

                "gt_box":best_gt["bbox"],

                "class_id":best_gt["class_id"],

                "class_name":best_gt["class_name"],

                "iou":best_iou

            })

        elif best_iou<0.3:

            negatives.append({

                "proposal_box":proposal_box,

                "iou":best_iou

            })

    training_samples[image_name]={

        "positive":positives,

        "negative":negatives

    }

    print(
        image_name,
        "Positive:",
        len(positives),
        "Negative:",
        len(negatives)
    )

with open(OUTPUT_FILE,"wb") as f:
    pickle.dump(training_samples,f)

print()
print("Saved",OUTPUT_FILE)

0.jpg Positive: 148 Negative: 2404
10.jpg Positive: 52 Negative: 824
100.jpg Positive: 183 Negative: 2485
1001.jpg Positive: 113 Negative: 2500
1004.jpg Positive: 105 Negative: 2770
1005.jpg Positive: 176 Negative: 2995
1006.jpg Positive: 123 Negative: 2569
1007.jpg Positive: 96 Negative: 3013
1008.jpg Positive: 86 Negative: 2666
1010.jpg Positive: 159 Negative: 2750

Saved training_samples.pkl


In [14]:
import os
import cv2
import pickle
import numpy as np
import tensorflow as tf

IMAGE_DIR="dataset/images/train"
TRAINING_SAMPLE_FILE="training_samples.pkl"
OUTPUT_FILE="features.pkl"

with open(TRAINING_SAMPLE_FILE,"rb") as f:
    training_samples=pickle.load(f)

cnn=tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

feature_dataset={}

for image_name in training_samples.keys():

    image=cv2.imread(
        os.path.join(
            IMAGE_DIR,
            image_name
        )
    )

    if image is None:
        continue

    image=cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    H,W=image.shape[:2]

    batch=[]

    information=[]
    for sample in training_samples[image_name]["positive"]:

        x1,y1,x2,y2=sample["proposal_box"]

        x1=max(0,x1)
        y1=max(0,y1)

        x2=min(W,x2)
        y2=min(H,y2)

        crop=image[
            y1:y2,
            x1:x2
        ]

        if crop.size==0:
            continue

        crop=cv2.resize(
            crop,
            (224,224)
        )

        crop=tf.keras.applications.resnet50.preprocess_input(
            crop.astype(np.float32)
        )

        batch.append(crop)

        information.append({

            "class_id":sample["class_id"],

            "proposal_box":sample["proposal_box"],

            "gt_box":sample["gt_box"],

            "iou":sample["iou"]

        })
    negatives=training_samples[image_name]["negative"]

    if len(negatives)>100:

        negatives=negatives[:100]

    for sample in negatives:

        x1,y1,x2,y2=sample["proposal_box"]

        x1=max(0,x1)
        y1=max(0,y1)

        x2=min(W,x2)
        y2=min(H,y2)

        crop=image[
            y1:y2,
            x1:x2
        ]

        if crop.size==0:
            continue

        crop=cv2.resize(
            crop,
            (224,224)
        )

        crop=tf.keras.applications.resnet50.preprocess_input(
            crop.astype(np.float32)
        )

        batch.append(crop)

        information.append({

            "class_id":-1,

            "proposal_box":sample["proposal_box"],

            "gt_box":None,

            "iou":sample["iou"]

        })
    batch=np.array(batch,dtype=np.float32)

    features=cnn.predict(
        batch,
        batch_size=32,
        verbose=0
    )

    image_features=[]

    for feature,info in zip(features,information):

        image_features.append({

            "feature":feature.flatten(),

            "class_id":info["class_id"],

            "proposal_box":info["proposal_box"],

            "gt_box":info["gt_box"],

            "iou":info["iou"]

        })

    feature_dataset[image_name]=image_features

    print(
        image_name,
        len(image_features)
    )

with open(OUTPUT_FILE,"wb") as f:

    pickle.dump(
        feature_dataset,
        f
    )

print()
print("Saved",OUTPUT_FILE)

0.jpg 248
10.jpg 152
100.jpg 283
1001.jpg 213
1004.jpg 205
1005.jpg 276
1006.jpg 223
1007.jpg 196
1008.jpg 186
1010.jpg 259

Saved features.pkl


In [16]:
import pickle
import joblib
import numpy as np
from sklearn.svm import LinearSVC

FEATURE_FILE="features.pkl"

with open(FEATURE_FILE,"rb") as f:
    feature_dataset=pickle.load(f)

classes=[0,1,2,3,4]

for target_class in classes:

    X=[]
    y=[]

    for image_name in feature_dataset.keys():

        for sample in feature_dataset[image_name]:

            X.append(sample["feature"])

            if sample["class_id"]==target_class:

                y.append(1)

            else:

                y.append(-1)

    X=np.array(X)
    y=np.array(y)

    print(
        "Training Class",
        target_class,
        "Samples",
        len(X)
    )
    positive=np.sum(y==1)
    negative=np.sum(y==-1)
    
    if positive==0 or negative==0:
        print(f"Skipping class {target_class}")
        continue
    svm=LinearSVC(
        C=1.0,
        max_iter=10000
    )

    svm.fit(
        X,
        y
    )

    model_name=f"svm_class_{target_class}.pkl"

    joblib.dump(
        svm,
        model_name
    )

    print(
        model_name,
        "saved"
    )

print()
print("Training Complete")

Training Class 0 Samples 2241
svm_class_0.pkl saved
Training Class 1 Samples 2241
svm_class_1.pkl saved
Training Class 2 Samples 2241
Skipping class 2
Training Class 3 Samples 2241
svm_class_3.pkl saved
Training Class 4 Samples 2241
svm_class_4.pkl saved

Training Complete


In [17]:
import pickle
import joblib
import numpy as np
from sklearn.linear_model import LinearRegression

FEATURE_FILE="features.pkl"
OUTPUT_FILE="bbox_regressor.pkl"

with open(FEATURE_FILE,"rb") as f:
    feature_dataset=pickle.load(f)

X=[]
Y=[]

for image_name in feature_dataset.keys():

    for sample in feature_dataset[image_name]:

        if sample["class_id"]==-1:
            continue

        px1,py1,px2,py2=sample["proposal_box"]
        gx1,gy1,gx2,gy2=sample["gt_box"]

        pw=px2-px1
        ph=py2-py1

        gw=gx2-gx1
        gh=gy2-gy1

        if pw<=0 or ph<=0:
            continue

        tx=(gx1-px1)/pw
        ty=(gy1-py1)/ph

        tw=np.log(gw/pw)
        th=np.log(gh/ph)

        X.append(sample["feature"])

        Y.append([
            tx,
            ty,
            tw,
            th
        ])

X=np.array(X)
Y=np.array(Y)

print("Training Samples:",len(X))

regressor=LinearRegression()

regressor.fit(
    X,
    Y
)

joblib.dump(
    regressor,
    OUTPUT_FILE
)

print("Saved",OUTPUT_FILE)

Training Samples: 1241
Saved bbox_regressor.pkl


In [21]:
import cv2
import joblib
import numpy as np
import tensorflow as tf
import pickle
import os
IMAGE_PATH="dataset/images/train/0.jpg"

class_names={
0:"Text",
1:"Title",
2:"List",
3:"Table",
4:"Figure"
}

cnn=tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

svms={}

for i in range(5):

    try:

        svms[i]=joblib.load(
            f"svm_class_{i}.pkl"
        )

    except:

        pass

bbox_regressor=joblib.load(
    "bbox_regressor.pkl"
)

image=cv2.imread(
    IMAGE_PATH
)

original=image.copy()

H,W=image.shape[:2]

ss=cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()

ss.setBaseImage(image)

ss.switchToSelectiveSearchFast()

rects=ss.process()

detections=[]
for (x,y,w,h) in rects[:500]:

    x1=max(0,x)
    y1=max(0,y)

    x2=min(W,x+w)
    y2=min(H,y+h)

    crop=image[
        y1:y2,
        x1:x2
    ]

    if crop.size==0:
        continue

    crop=cv2.resize(
        crop,
        (224,224)
    )

    crop=tf.keras.applications.resnet50.preprocess_input(
        crop.astype(np.float32)
    )

    crop=np.expand_dims(
        crop,
        axis=0
    )

    feature=cnn.predict(
        crop,
        verbose=0
    ).flatten()

    best_score=-100000
    best_class=-1

    for cls in svms.keys():

        score=svms[cls].decision_function(
            [feature]
        )[0]

        if score>best_score:

            best_score=score
            best_class=cls

    if best_score<0:

        continue

    tx,ty,tw,th=bbox_regressor.predict(
        [feature]
    )[0]

    pw=x2-x1
    ph=y2-y1

    nx=x1+tx*pw
    ny=y1+ty*ph

    nw=pw*np.exp(tw)
    nh=ph*np.exp(th)

    detections.append({

        "class":best_class,

        "score":best_score,

        "bbox":[
            int(nx),
            int(ny),
            int(nx+nw),
            int(ny+nh)
        ]

    })
    

boxes=[]
scores=[]
prediction_results={

    os.path.basename(IMAGE_PATH):[]

}
for detection in detections:

    boxes.append([

        detection["bbox"][0],
        detection["bbox"][1],
        detection["bbox"][2]-detection["bbox"][0],
        detection["bbox"][3]-detection["bbox"][1]

    ])

    scores.append(float(detection["score"]))

indices=cv2.dnn.NMSBoxes(
    boxes,
    scores,
    score_threshold=0.0,
    nms_threshold=0.3
)

if len(indices)>0:

    for idx in indices:

        if isinstance(idx,(list,tuple,np.ndarray)):
            idx=idx[0]

        detection=detections[idx]
        prediction_results[os.path.basename(IMAGE_PATH)].append({

                "class":detection["class"],
            
                "bbox":detection["bbox"],
            
                "score":float(detection["score"])
            
            })

        x1,y1,x2,y2=detection["bbox"]

        cv2.rectangle(
            original,
            (x1,y1),
            (x2,y2),
            (0,255,0),
            2
        )

        cv2.putText(
            original,
            class_names[detection["class"]],
            (x1,max(20,y1-5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,0,255),
            2
        )

        print(
            class_names[detection["class"]],
            round(detection["score"],2),
            detection["bbox"]
        )

cv2.imshow(
    "RCNN Detection",
    original
)

cv2.waitKey(0)

cv2.destroyAllWindows()
with open("predictions.pkl","wb") as f:

    pickle.dump(
        prediction_results,
        f
    )

print("Saved predictions.pkl")

Text 3.66 [55, 278, 556, 297]
Text 2.74 [56, 517, 304, 527]
Text 2.66 [56, 183, 513, 212]
Text 2.56 [56, 344, 297, 443]
Text 2.2 [315, 556, 556, 724]
Text 2.17 [55, 568, 296, 696]
Text 2.1 [-3, 202, 242, 221]
Title 2.01 [176, 121, 525, 135]
Title 1.91 [346, 89, 422, 112]
Text 1.88 [56, 731, 361, 740]
Text 1.81 [56, 453, 297, 506]
Title 1.78 [55, 555, 121, 568]
Text 1.74 [316, 335, 555, 410]
Title 1.63 [107, 91, 519, 117]
Title 1.25 [346, 87, 420, 95]
Text 1.11 [314, 443, 554, 507]
Text 1.0 [47, 396, 588, 555]
Text 0.77 [30, 222, 201, 234]
Text 0.63 [3, 241, 646, 286]
Title 0.5 [442, 114, 498, 118]
Title 0.19 [404, 9, 552, 57]
Text 0.17 [-7, 102, 599, 132]
Text 0.14 [379, 42, 537, 44]
Title 0.09 [412, 38, 580, 78]
Title 0.08 [47, 67, 620, 109]
Title 0.04 [235, 69, 316, 81]
Saved predictions.pkl


In [22]:
import pickle
import numpy as np

GROUND_TRUTH_FILE="ground_truth.pkl"
PREDICTION_FILE="predictions.pkl"

with open(GROUND_TRUTH_FILE,"rb") as f:
    ground_truth=pickle.load(f)

with open(PREDICTION_FILE,"rb") as f:
    predictions=pickle.load(f)

def iou(box1,box2):

    x1=max(box1[0],box2[0])
    y1=max(box1[1],box2[1])

    x2=min(box1[2],box2[2])
    y2=min(box1[3],box2[3])

    inter=max(0,x2-x1)*max(0,y2-y1)

    area1=(box1[2]-box1[0])*(box1[3]-box1[1])
    area2=(box2[2]-box2[0])*(box2[3]-box2[1])

    union=area1+area2-inter

    if union==0:
        return 0

    return inter/union

TP=0
FP=0
FN=0

for image_name in predictions.keys():

    gt=ground_truth[image_name]["objects"]

    pred=predictions[image_name]

    matched=[]

    for p in pred:

        found=False

        for i,g in enumerate(gt):

            if i in matched:
                continue

            if p["class"]!=g["class_id"]:
                continue

            if iou(
                p["bbox"],
                g["bbox"]
            )>=0.5:

                TP+=1
                matched.append(i)
                found=True
                break

        if not found:
            FP+=1

    FN+=len(gt)-len(matched)

precision=TP/(TP+FP+1e-8)

recall=TP/(TP+FN+1e-8)

f1=2*precision*recall/(precision+recall+1e-8)

print("TP:",TP)
print("FP:",FP)
print("FN:",FN)

print()

print("Precision:",round(precision,4))
print("Recall:",round(recall,4))
print("F1 Score:",round(f1,4))

TP: 11
FP: 15
FN: 3

Precision: 0.4231
Recall: 0.7857
F1 Score: 0.55


In [ ]:
import os

files=[
    "dataset.py",
    "selective_search.py",
    "iou.py",
    "feature_extractor.py",
    "svm_train.py",
    "bbox_regressor.py",
    "inference.py",
    "evaluate.py"
]

for file in files:

    print("\n"+"="*60)
    print("Running:",file)
    print("="*60)

    status=os.system(f"python {file}")

    if status!=0:

        print("Error while running",file)

        break

print("\nPipeline Finished")